## Validation of Exploratory Persistence Against Published Differential-Expression Results

### **Purpose**

#### This notebook evaluates whether the recovery and persistence patterns identified in the exploratory analysis are consistent with the published differential-expression results from the same RNA-seq experiment.

#### The earlier analysis used rlog-transformed expression values and a control-derived empirical response floor to classify gene-level recovery behavior, including persistent_same_direction responses. Because this approach is descriptive rather than a formal differential-expression test, the published DEG results provide an opportunity to assess whether the same biological signal is also supported by the study's count-based statistical analysis.

#### The analysis first examines the structure and filtering of the supplementary DEG workbook, confirms the available endpoint and recovery comparisons, and records the conditions for which exact recovery DEG lists are available. Published recovery DEGs are then compared with the exploratory persistent-gene sets to evaluate:

#### - gene-level overlap between the two approaches;

#### - coverage of published DEGs within the exploratory informative-gene universe;

#### - whether the observed overlap exceeds that expected by chance using Fisher's exact test;

#### - the strength of overlap enrichment across recovery conditions;

#### - statistical significance after Benjamini-Hochberg FDR correction; and

#### - agreement in expression direction after accounting for the opposite contrast orientation used in the published workbook.

#### Particular attention is given to drought, where both analyses identify a substantial residual transcriptional response after recovery.

#### This notebook provides methodological validation within the same experiment, rather than independent biological replication. Its purpose is therefore to determine whether the exploratory persistence framework captures a recovery-stage expression signal that is consistent with the study's formal differential-expression analysis while preserving the distinction between exploratory persistence and statistical DEG significance.

### Loading and Inspecting the Published Differential-Expression Workbook

#### The published supplementary differential-expression workbook was located and opened using pandas. File existence was checked before loading to confirm that the expected workbook could be accessed from the project directory.

#### The workbook was initially inspected through its worksheet names rather than loading an assumed comparison directly. This allows the available published analyses and contrasts to be identified before determining how they can be matched to the stress, endpoint, recovery, and control comparisons used in the current project.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Display settings
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
pd.set_option("display.precision", 3)

# Plot defaults
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.bbox"] = "tight"

In [3]:
DEG_WORKBOOK = Path(
    "../data/external/GSE226105_supplementary/mmc16.xlsx"
)

DEG_WORKBOOK.exists()

True

In [4]:
deg_workbook = pd.ExcelFile(DEG_WORKBOOK)

deg_workbook.sheet_names

['Table S3',
 'HL rec',
 'HL ep',
 'Humidity rec',
 'Humidity ep',
 'Drought rec',
 'Drought ep',
 'Heat ep',
 'Heat drought rec',
 'Heat drought ep',
 'Heat HL ep',
 'Heat drought hl ep',
 'Cold rec',
 'Cold ep',
 'Cold HL ep']

### **Observation —** Structure of the Published Differential-Expression Workbook

#### The supplementary workbook contains 15 worksheets:

#### - 1 general worksheet

#### - 9 endpoint worksheets, covering all nine experimental stress conditions

#### - 5 recovery worksheets, covering only a subset of the stress conditions

#### Endpoint DEG results are available for all nine stress conditions.

#### Recovery DEG results are available only for:

#### `high_light`, `high_humidity`, `drought`, `heat_drought`, and `cold`

#### No recovery DEG sheet is present for:

#### `heat`, `heat_high_light`, `heat_drought_high_light`, or `cold_high_light`

#### The published supplementary workbook contains complete endpoint DEG results across all nine stress conditions but recovery DEG results for only five conditions. This creates an asymmetric validation structure: endpoint-based comparisons may potentially be evaluated across the complete experimental stress set, whereas recovery-based validation will be limited to the conditions for which published recovery results are available.

#### The meaning of the missing recovery sheets is not assumed at this stage. The general sheet and the structure of the individual DEG worksheets should be inspected before deciding how the published comparisons correspond to the project's endpoint and recovery analyses.

### Building a Manifest for the Published DEG Sheets

#### The published DEG worksheets were mapped to standardized experimental states and stress-condition names using a manifest table. The original worksheet names were preserved so that each sheet can later be loaded directly from the supplementary workbook, while the state and condition columns provide labels consistent with those used throughout the current project.

#### The manifest contains the 14 condition-specific DEG worksheets identified in the workbook: **nine endpoint sheets** and **five recovery sheets**. Grouping the manifest by experimental state provides a simple validation that the expected endpoint and recovery sheet counts have been represented correctly.

In [5]:
published_de_manifest = pd.DataFrame(
    [
        ("HL ep", "endpoint", "high_light"),
        ("Humidity ep", "endpoint", "high_humidity"),
        ("Drought ep", "endpoint", "drought"),
        ("Heat ep", "endpoint", "heat"),
        ("Heat drought ep", "endpoint", "heat_drought"),
        ("Heat HL ep", "endpoint", "heat_high_light"),
        ("Heat drought hl ep", "endpoint", "heat_drought_high_light"),
        ("Cold ep", "endpoint", "cold"),
        ("Cold HL ep", "endpoint", "cold_high_light"),

        ("HL rec", "recovery", "high_light"),
        ("Humidity rec", "recovery", "high_humidity"),
        ("Drought rec", "recovery", "drought"),
        ("Heat drought rec", "recovery", "heat_drought"),
        ("Cold rec", "recovery", "cold"),
    ],
    columns=[
        "sheet_name",
        "state",
        "condition",
    ],
)

published_de_manifest

,sheet_name,state,condition
0,HL ep,endpoint,high_light
1,Humidity ep,endpoint,high_humidity
2,Drought ep,endpoint,drought
3,Heat ep,endpoint,heat
4,Heat drought ep,endpoint,heat_drought
5,Heat HL ep,endpoint,heat_high_light
6,Heat drought hl ep,endpoint,heat_drought_high_light
7,Cold ep,endpoint,cold
8,Cold HL ep,endpoint,cold_high_light
9,HL rec,recovery,high_light


### Validating the Number of Published DEG Sheets by Experimental State

In [6]:
published_de_manifest.groupby("state").size()

state
endpoint    9
recovery    5
dtype: int64

#### The manifest is correct: 9 endpoint sheets, 5 recovery sheets, with condition names normalized exactly the way we need.

### Inspecting the Structure of a Published Endpoint DEG Sheet

#### A representative published endpoint DEG sheet `Drought ep` was loaded and inspected before attempting to process all DEG worksheets automatically. The first rows, table dimensions, and column names were examined to determine how the published differential-expression results are structured and which variables are available for validation.

#### This inspection is necessary before interpreting row counts or fold-change directions, because the meaning of the table and the orientation of the published comparison must be confirmed from the supplied results or accompanying documentation before downstream comparisons are made.

In [7]:
drought_ep = pd.read_excel(
    DEG_WORKBOOK,
    sheet_name="Drought ep"
)

drought_ep.head()

,gene_ID,base,log2(FC),sd,Wald-test,P,adj-P
0,AT5G44020,2790.290,5.088,0.148,34.282,1.441e-257,3.090e-253
1,AT5G65730,993.309,5.141,0.178,28.859,3.956e-183,4.241e-179
2,AT2G46680,4517.464,-5.727,0.206,-27.788,6.069e-170,4.338e-166
3,AT3G53980,2291.677,-6.858,0.251,-27.343,1.300e-164,6.970e-161
4,AT5G03210,1528.888,-7.654,0.313,-24.462,3.729e-132,1.599e-128


In [8]:
drought_ep.shape

(5275, 7)

In [9]:
drought_ep.columns.tolist()

['gene_ID', 'base', 'log2(FC)', 'sd', 'Wald-test', 'P', 'adj-P']

### **Observation —** Published Drought Endpoint DEG Table

#### The Drought ep worksheet contains 5,275 rows and 7 columns, with fields for `gene_ID`, `base`, `log2(FC)`, `sd`, `Wald-test`, `P`, and `adj-P`.

#### The table therefore contains gene-level fold-change estimates together with statistical information from the published differential-expression analysis. Both large positive and large negative log2(FC) values are present, showing that the published results include substantial expression changes in both directions.

#### The exact biological interpretation of positive and negative fold-change values is deferred until the orientation of the published contrast has been confirmed.

### Validating the Structure and Filtering of the Published Drought Endpoint DEG Table

#### The Drought ep worksheet was checked for missing values, duplicate gene identifiers, and the range of fold-change and adjusted-p-value values. Gene-ID uniqueness was examined to confirm whether the table has a one-row-per-gene structure.

#### The minimum absolute log2(FC) and maximum adj-P were also calculated, together with Boolean checks for whether all reported genes satisfy `|log2(FC)| ≥ 1` and `adj-P ≤ 0.05`. These checks help determine whether the supplementary worksheet appears to contain an already filtered set of published DEGs rather than the complete set of genes tested in the differential-expression model.

In [10]:
drought_ep.isna().sum()

gene_ID      0
base         0
log2(FC)     0
sd           0
Wald-test    0
P            0
adj-P        0
dtype: int64

In [11]:
(
    drought_ep["gene_ID"].nunique(),
    drought_ep["gene_ID"].duplicated().sum()
)

(5275, np.int64(0))

In [12]:
drought_ep_validation = {                                             # compact validation summary
    "n_genes": len(drought_ep),
    "min_abs_log2fc": drought_ep["log2(FC)"].abs().min(),
    "max_adj_p": drought_ep["adj-P"].max(),
    "all_abs_log2fc_ge_1": (
        drought_ep["log2(FC)"].abs() >= 1
    ).all(),
    "all_adj_p_le_0_05": (
        drought_ep["adj-P"] <= 0.05
    ).all(),
}

drought_ep_validation

{'n_genes': 5275,
 'min_abs_log2fc': np.float64(1.00004238751055),
 'max_adj_p': np.float64(0.0499993628090402),
 'all_abs_log2fc_ge_1': np.True_,
 'all_adj_p_le_0_05': np.True_}

### **Observation —** Published DEG Table Structure

#### The Drought ep worksheet contains 5,275 unique genes, with no missing values or duplicate gene identifiers. 

#### Every listed gene has an absolute log2 fold change of at least 1 and an adjusted p-value of at most 0.05. 

#### The minimum absolute log2(FC) is **1.00004** and the maximum adj-P is **0.049999**, indicating that this worksheet contains a pre-filtered set of significant DEGs rather than the complete differential-expression output for all tested genes.

### Loading and Standardizing All Published DEG Worksheets

#### The 14 condition-specific published DEG worksheets were loaded into a dictionary and standardized to a common column structure where necessary. 

#### A summary table was then constructed to validate each worksheet for gene-ID uniqueness, duplicate and missing values, minimum absolute log2 fold change, maximum adjusted p-value, and consistency with the apparent `|log2FC| ≥ 1` and `adj-P ≤ 0.05` filtering boundaries.

#### This extends the validation performed on the `Drought ep` worksheet to the complete published DEG collection and allows differences in worksheet structure or filtering behavior to be identified before the published results are compared with the exploratory analysis.

In [13]:
published_de_tables = {}

for _, row in published_de_manifest.iterrows():
    sheet = row["sheet_name"]

    df = pd.read_excel(
        DEG_WORKBOOK,
        sheet_name=sheet
    )

    if sheet == "Heat HL ep":
        df = df.rename(
            columns={
                "FC": "log2(FC)",
                "pval": "P",
                "FDR": "adj-P",
            }
        )

    published_de_tables[sheet] = df

In [14]:
published_de_summary = []                                     # summarizing and validating all published DEG tables

for _, row in published_de_manifest.iterrows():

    sheet = row["sheet_name"]
    df = published_de_tables[sheet]

    published_de_summary.append(
        {
            "sheet_name": sheet,
            "state": row["state"],
            "condition": row["condition"],
            "n_genes": len(df),
            "n_unique_genes": df["gene_ID"].nunique(),
            "n_duplicate_genes": df["gene_ID"].duplicated().sum(),
            "n_missing_values": df.isna().sum().sum(),
            "min_abs_log2fc": df["log2(FC)"].abs().min(),
            "max_adj_p": df["adj-P"].max(),
            "all_abs_log2fc_ge_1": (
                df["log2(FC)"].abs() >= 1
            ).all(),
            "all_adj_p_le_0_05": (
                df["adj-P"] <= 0.05
            ).all(),
        }
    )

published_de_summary = pd.DataFrame(
    published_de_summary
)

published_de_summary

,sheet_name,state,condition,n_genes,n_unique_genes,n_duplicate_genes,n_missing_values,min_abs_log2fc,max_adj_p,all_abs_log2fc_ge_1,all_adj_p_le_0_05
0,HL ep,endpoint,high_light,55,55,0,0,1.001,2.840e-04,True,True
1,Humidity ep,endpoint,high_humidity,249,249,0,0,1.002,1.587e-03,True,True
2,Drought ep,endpoint,drought,5275,5275,0,0,1.000,5.000e-02,True,True
3,Heat ep,endpoint,heat,21,21,0,0,1.005,6.080e-06,True,True
4,Heat drought ep,endpoint,heat_drought,8114,8114,0,0,1.000,4.963e-02,True,True
5,Heat HL ep,endpoint,heat_high_light,7,7,0,0,1.003,1.869e-06,True,True
6,Heat drought hl ep,endpoint,heat_drought_high_light,1392,1392,0,0,1.000,4.995e-02,True,True
7,Cold ep,endpoint,cold,960,960,0,0,1.001,6.515e-03,True,True
8,Cold HL ep,endpoint,cold_high_light,6632,6632,0,0,1.000,4.997e-02,True,True
9,HL rec,recovery,high_light,2,2,0,0,1.004,1.732e-02,True,True


### **Observation —** Published DEG Workbook Validation

#### Across the 14 DEG worksheets, gene identifiers were unique and no missing values were detected. 

#### Thirteen worksheets contained only genes with an absolute log2 fold change of at least 1 and an adjusted p-value of at most 0.05. 

#### The Heat drought rec worksheet contained one borderline entry with an absolute log2 fold change of 0.998, while still satisfying the adjusted p-value criterion. This small boundary deviation was retained without modification and does not provide strong evidence that the worksheet follows a different filtering rule. 

#### Overall, the workbook is consistent with a collection of pre-filtered significant DEG lists rather than complete differential-expression outputs.

### Inspecting the Borderline Heat + Drought Recovery DEG Entry

#### The Heat drought rec worksheet contained the only observed entry with an absolute log2 fold change slightly below 1. 

#### The row with the minimum absolute log2(FC) was therefore inspected directly to identify the gene and examine its complete published differential-expression statistics. The original published value was retained unchanged.

In [15]:
heat_drought_rec = published_de_tables["Heat drought rec"]

heat_drought_rec.loc[
    heat_drought_rec["log2(FC)"].abs().idxmin()
]

gene_ID      AT2G32160
base            694.38
log2(FC)        -0.998
sd               0.217
Wald-test       -4.592
P                  0.0
adj-P            0.035
Name: 2, dtype: object

#### The minimum absolute fold change in the Heat drought rec worksheet corresponds to AT2G32160, with:

#### - log2(FC) = -0.998038

#### - adj-P    = 0.0347

#### The gene satisfies the published adjusted-p-value threshold but falls short of an absolute log2 fold change of 1 by only about 0.002. The published value is therefore retained unchanged and recorded as a small boundary discrepancy rather than being filtered or corrected.

### Summarizing Published DEG Counts by Condition and Experimental State

#### The published DEG summary was reshaped so that endpoint and recovery DEG counts could be compared side by side for each stress condition. Conditions without a published recovery worksheet appear as missing recovery values rather than zero counts.

#### A Boolean `recovery_sheet_present` field was therefore added to explicitly distinguish conditions with published recovery DEG results from conditions for which no recovery worksheet is available. This prevents missing recovery analyses from being incorrectly interpreted as conditions with zero recovery DEGs.

In [16]:
published_de_counts = (
    published_de_summary[
        ["state", "condition", "n_genes"]
    ]
    .pivot(
        index="condition",
        columns="state",
        values="n_genes"
    )
    .reset_index()
)

published_de_counts

state,condition,endpoint,recovery
0,cold,960.0,5.0
1,cold_high_light,6632.0,NaN
2,drought,5275.0,567.0
3,heat,21.0,NaN
4,heat_drought,8114.0,3.0
5,heat_drought_high_light,1392.0,NaN
6,heat_high_light,7.0,NaN
7,high_humidity,249.0,2.0
8,high_light,55.0,2.0


### Recording Whether a Published Recovery DEG Sheet Is Available

#### This step preserves the distinction between an unavailable recovery DEG count and a true count of zero, preventing missing values from being inadvertently recoded as 0 and misinterpreted as evidence of zero published recovery DEGs.

In [17]:
published_de_counts["recovery_sheet_present"] = (
    published_de_counts["recovery"].notna()
)

published_de_counts

state,condition,endpoint,recovery,recovery_sheet_present
0,cold,960.0,5.0,True
1,cold_high_light,6632.0,NaN,False
2,drought,5275.0,567.0,True
3,heat,21.0,NaN,False
4,heat_drought,8114.0,3.0,True
5,heat_drought_high_light,1392.0,NaN,False
6,heat_high_light,7.0,NaN,False
7,high_humidity,249.0,2.0,True
8,high_light,55.0,2.0,True


### **Observation —** Availability of Published Recovery DEG Results

#### Recovery samples were available for all nine stress conditions, and the GEO processing description indicates that differential-expression comparisons were performed for both endpoint and recovery treatments against standard conditions. However, recovery DEG worksheets are present in the supplementary workbook for only five conditions.

#### The accompanying study reports that fewer than 10 significant DEGs were detected after recovery for each stress condition except drought, which retained substantially more.

#### The absence of a recovery worksheet therefore does not indicate that the corresponding recovery contrast was not performed. Instead, it is consistent with the study reporting very few significant recovery DEGs for those conditions.

#### Because “fewer than 10” does not establish an exact count of zero, missing recovery worksheets should remain recorded as worksheet absent / exact published DEG count unavailable unless the paper or supplementary material explicitly provides the condition-specific count. This preserves the difference between the experimental result and the way the result was represented in the workbook.

### Preserving the Missing Recovery Counts Without Converting Them to Zero

#### Published endpoint DEG counts were stored as integers, while recovery DEG counts were kept as nullable integers so that conditions without an exact worksheet count remained explicitly missing. 

#### A separate status field was added to distinguish recovery conditions with an exact published DEG count from those for which the workbook provides no worksheet but the study reports fewer than 10 recovery DEGs. This preserves both the numerical information and its reporting context.

In [18]:
published_de_counts["published_recovery_degs"] = (
    published_de_counts["recovery"]
    .astype("Int64")
)

published_de_counts["endpoint"] = (
    published_de_counts["endpoint"]
    .astype(int)
)

published_de_counts["recovery_count_status"] = np.where(
    published_de_counts["recovery_sheet_present"],
    "exact count available",
    "exact count unavailable; paper reports <10"
)

published_de_counts[
    [
        "condition",
        "endpoint",
        "published_recovery_degs",
        "recovery_sheet_present",
        "recovery_count_status",
    ]
]

state,condition,endpoint,published_recovery_degs,recovery_sheet_present,recovery_count_status
0,cold,960,5,True,exact count available
1,cold_high_light,6632,<NA>,False,exact count unavailable; paper reports <10
2,drought,5275,567,True,exact count available
3,heat,21,<NA>,False,exact count unavailable; paper reports <10
4,heat_drought,8114,3,True,exact count available
5,heat_drought_high_light,1392,<NA>,False,exact count unavailable; paper reports <10
6,heat_high_light,7,<NA>,False,exact count unavailable; paper reports <10
7,high_humidity,249,2,True,exact count available
8,high_light,55,2,True,exact count available


### **Observation —** Published Endpoint and Recovery DEG Counts

#### The published endpoint DEG counts vary substantially across stress conditions, ranging from 7 genes for heat + high light to 8,114 genes for heat + drought. 

#### Exact recovery DEG counts are available for five conditions: cold (5), drought (567), heat + drought (3), high humidity (2), and high light (2).

#### Drought is clearly different from the other recovery conditions with exact counts, retaining 567 significant DEGs, whereas each of the other four contains only 2–5. 

#### For `cold_high_light`, `heat`, `heat_drought_high_light`, and `heat_high_light`, the exact recovery DEG count remains unavailable because no corresponding worksheet is present; these values are therefore retained as missing rather than being assigned a count of zero.

### Loading the Exploratory Gene Recovery Categories

#### Loading our previously assigned gene-level recovery categories so they can be compared with the published differential-expression results.

In [19]:
RECOVERY_CATEGORIES_PATH = Path(
    "../data/interim/exploratory_analysis/gene_recovery_categories.csv"
)

recovery_categories = pd.read_csv(
    RECOVERY_CATEGORIES_PATH
)

recovery_categories.shape

(26118, 10)

In [20]:
recovery_categories.columns.tolist()

['gene_id',
 'cold',
 'cold_high_light',
 'drought',
 'heat',
 'heat_drought',
 'heat_drought_high_light',
 'heat_high_light',
 'high_humidity',
 'high_light']

In [21]:
recovery_categories.head()

,gene_id,cold,cold_high_light,drought,heat,heat_drought,heat_drought_high_light,heat_high_light,high_humidity,high_light
0,AT1G01010,NaN,recovered,recovered,NaN,recovered,recovered,NaN,NaN,NaN
1,AT1G01020,recovered,recovered,recovered,NaN,recovered,recovered,NaN,NaN,NaN
2,AT1G01030,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AT1G01040,NaN,recovered,recovered,NaN,recovered,NaN,NaN,NaN,NaN
4,AT1G01050,recovered,recovered,recovered,NaN,recovered,recovered,NaN,NaN,recovered


### Comparing Exploratory Persistent Genes with Published Recovery DEGs

#### Exploratory persistent_same_direction genes were converted into condition-specific gene sets and compared with the published recovery DEG sets for the five conditions with available recovery worksheets. Conditions without an exact published recovery gene list were retained as unavailable and excluded from gene-level overlap calculations rather than being treated as empty DEG sets.

#### For each comparable condition, the analysis records the number of exploratory persistent genes, published recovery DEGs, and genes shared by both methods.

#### Two complementary overlap fractions were calculated: the proportion of exploratory persistent genes supported by the published recovery DEG set, and the proportion of published recovery DEGs captured by the exploratory persistence classification. These measures use different denominators and therefore answer different validation questions.

In [22]:
conditions = recovery_categories.columns.drop("gene_id")

our_persistent_sets = {                                                        # building the exploratory persistent gene sets by condition
    condition: set(
        recovery_categories.loc[
            recovery_categories[condition] == "persistent_same_direction",
            "gene_id"
        ]
        .astype(str)
        .str.strip()
    )
    for condition in conditions
}

In [23]:
{
    condition: len(genes)
    for condition, genes in our_persistent_sets.items()
}

{'cold': 71,
 'cold_high_light': 187,
 'drought': 1240,
 'heat': 70,
 'heat_drought': 344,
 'heat_drought_high_light': 255,
 'heat_high_light': 72,
 'high_humidity': 28,
 'high_light': 779}

In [24]:
published_recovery_sets = {}                                                  # building the published recovery-DEG sets by condition

for condition in conditions:

    matching_rows = published_de_manifest[
        (published_de_manifest["state"] == "recovery")
        & (published_de_manifest["condition"] == condition)
    ]

    if len(matching_rows) == 1:

        sheet = matching_rows.iloc[0]["sheet_name"]

        published_recovery_sets[condition] = set(
            published_de_tables[sheet]["gene_ID"]
            .astype(str)
            .str.strip()
        )

    else:

        published_recovery_sets[condition] = None

In [25]:
{
    condition: (
        len(genes)
        if genes is not None
        else pd.NA
    )
    for condition, genes in published_recovery_sets.items()
}

{'cold': 5,
 'cold_high_light': <NA>,
 'drought': 567,
 'heat': <NA>,
 'heat_drought': 3,
 'heat_drought_high_light': <NA>,
 'heat_high_light': <NA>,
 'high_humidity': 2,
 'high_light': 2}

In [26]:
validation_rows = []                                                         # comparing the exploratory persistence with published recovery DEGs

for condition in conditions:

    our_genes = our_persistent_sets[condition]
    published_genes = published_recovery_sets[condition]

    if published_genes is None:
        continue

    shared_genes = our_genes & published_genes

    validation_rows.append(
        {
            "condition": condition,
            "our_persistent_genes": len(our_genes),
            "published_recovery_degs": len(published_genes),
            "shared_genes": len(shared_genes),
            "fraction_of_our_persistent_formally_de": (
                len(shared_genes) / len(our_genes)
                if len(our_genes) > 0
                else np.nan
            ),
            "fraction_of_published_degs_captured": (
                len(shared_genes) / len(published_genes)
                if len(published_genes) > 0
                else np.nan
            ),
        }
    )

published_de_validation = pd.DataFrame(
    validation_rows
)

published_de_validation

,condition,our_persistent_genes,published_recovery_degs,shared_genes,fraction_of_our_persistent_formally_de,fraction_of_published_degs_captured
0,cold,71,5,2,0.028,0.400
1,drought,1240,567,348,0.281,0.614
2,heat_drought,344,3,2,0.006,0.667
3,high_humidity,28,2,1,0.036,0.500
4,high_light,779,2,1,0.001,0.500


### **Insight —** Comparison with Published Recovery Differential Expression

#### The largest gene-level overlap between the exploratory persistence framework and the published recovery differential-expression results occurs under drought. Of the 1,240 genes classified as persistent_same_direction after drought, 348 were also present among the 567 published recovery DEGs. The exploratory framework therefore captured approximately 61% of the published drought recovery-DEG set, while approximately 28% of the broader exploratory persistent set also met the more stringent published DEG criteria.

#### For the other conditions with available recovery worksheets, the published DEG sets were very small: 2 of 5 cold recovery DEGs, 2 of 3 heat+drought recovery DEGs, and 1 of 2 recovery DEGs under both high humidity and high light overlapped with the exploratory persistent sets. Because these published sets contain only a few genes, their overlap fractions should be interpreted cautiously.

#### The incomplete overlap is expected because the two approaches define persistence differently. The exploratory framework identifies genes whose recovery expression remains beyond a control-derived empirical response floor in the same direction as the endpoint response, whereas the published differential-expression analysis requires approximately `|log2FC| ≥ 1` together with statistical support after multiple-testing correction. The comparison therefore shows substantial agreement for a subset of genes—most clearly under drought—while also demonstrating that the exploratory persistence criterion captures a broader class of residual expression responses than the published DEG threshold.

### Checking Coverage of Published Recovery DEGs in the Exploratory Informative-Gene Universe

#### The published recovery DEG sets were compared with the informative-gene universe used in the exploratory analysis. This determines whether published recovery DEGs that were not classified as persistent were actually eligible to be detected by the exploratory framework.

#### For each recovery condition with an available published gene list, the number of published DEGs present within and outside the exploratory informative-gene universe was recorded. Conditions without an exact published recovery gene list remain unavailable rather than being assigned inferred gene counts.

In [27]:
informative_gene_ids = set(
    recovery_categories["gene_id"]
    .astype(str)
    .str.strip()
)

published_recovery_coverage = []

for condition in conditions:

    published_genes = published_recovery_sets[condition]

    if published_genes is None:

        published_recovery_coverage.append(
            {
                "condition": condition,
                "published_recovery_degs": pd.NA,
                "present_in_our_informative_genes": pd.NA,
                "outside_our_informative_genes": pd.NA,
            }
        )

        continue

    genes_in_our_universe = (
        published_genes & informative_gene_ids
    )

    published_recovery_coverage.append(
        {
            "condition": condition,
            "published_recovery_degs": len(published_genes),
            "present_in_our_informative_genes": len(
                genes_in_our_universe
            ),
            "outside_our_informative_genes": (
                len(published_genes)
                - len(genes_in_our_universe)
            ),
        }
    )

published_recovery_coverage = pd.DataFrame(
    published_recovery_coverage
)

published_recovery_coverage

,condition,published_recovery_degs,present_in_our_informative_genes,outside_our_informative_genes
0,cold,5,5,0
1,cold_high_light,<NA>,<NA>,<NA>
2,drought,567,567,0
3,heat,<NA>,<NA>,<NA>
4,heat_drought,3,3,0
5,heat_drought_high_light,<NA>,<NA>,<NA>
6,heat_high_light,<NA>,<NA>,<NA>
7,high_humidity,2,2,0
8,high_light,2,2,0


### **Observation —** Coverage of Published Recovery DEGs in the Exploratory Gene Universe

#### All 579 published recovery DEGs with available gene-level lists were present in the informative-gene universe used for the exploratory analysis. This includes 5/5 genes under cold, 567/567 under drought, 3/3 under heat+drought, and 2/2 under both high humidity and high light. No published recovery DEG from these five conditions fell outside the exploratory analysis universe.

#### For recovery conditions without a supplementary DEG worksheet, gene-level coverage remains unavailable because the exact published DEG identities are not provided.

### **Insight —** Gene-Universe Coverage Does Not Explain the Incomplete Overlap

#### The incomplete agreement between exploratory persistent genes and published recovery DEGs cannot be explained by differences in gene coverage. Every published recovery DEG available for comparison was included in the exploratory informative-gene universe and therefore had the opportunity to be classified by our framework.

#### The remaining disagreement must therefore arise from differences in how the two approaches define and detect persistent expression, including the exploratory control-derived response floor versus the published fold-change and statistical-significance requirements.

### Testing Whether Exploratory Persistent Genes Overlap Published Recovery DEGs More Than Expected by Chance

#### Condition-specific overlap between exploratory `persistent_same_direction` genes and published recovery DEGs was tested using a one-sided Fisher exact test against the 26,118-gene informative universe. For each condition, the observed overlap was compared with the overlap expected under random set membership, and both overlap enrichment and the Fisher odds ratio were retained as measures of association strength.

#### The Fisher test evaluates whether published recovery DEGs are overrepresented among exploratory persistent genes rather than merely overlapping by chance.

#### Raw p-values from the five available recovery comparisons were subsequently corrected using the Benjamini–Hochberg procedure to control the false discovery rate across conditions.

In [28]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

### **Note —** Fisher Exact-Test Contingency Table

#### For each stress condition, every gene in the 26,118-gene informative universe is placed into one of four groups:

|                                      | Published recovery DEG | Not a published recovery DEG |
|--------------------------------------|------------------------|------------------------------|
| **Exploratory persistent gene**      | **a**                  | **b**                        |
| **Not an exploratory persistent gene** | **c**                | **d**                        |


#### The four values mean:

#### - a = genes found by both methods

#### - b = genes classified as persistent by our exploratory method but not present in the published recovery-DEG set

#### - c = published recovery DEGs not classified as persistent by our exploratory method

#### - d = informative genes found in neither set

#### They are calculated as:

#### a = shared genes

#### b = our persistent genes only

#### c = published recovery DEGs only

#### d = all remaining informative genes

#### Together:

#### a + b + c + d = 26,118

#### This gives the 2 × 2 table required for Fisher's exact test.

### **Note —** Expected Gene Overlap and Fisher Enrichment Testing

#### The 26,118 informative genes are used as the background because these are the genes that were eligible to receive an exploratory recovery classification.

#### The expected overlap by chance is calculated as:

#### Expected overlap = (number of exploratory persistent genes × number of published recovery DEGs) / number of informative genes

#### This estimates how many genes the two sets would be expected to share if membership in the exploratory persistent set and the published recovery-DEG set were unrelated.

#### The observed number of shared genes is then compared with this expected overlap. The overlap_enrichment value shows how many times larger the observed overlap is than the overlap expected by chance.

#### `alternative="greater"` is used in Fisher's exact test because the specific question is whether published recovery DEGs are overrepresented among the exploratory persistent genes.

#### The Fisher p-value measures the statistical evidence for this enrichment. Because the test is performed separately for five recovery conditions, the resulting p-values are subsequently corrected using the Benjamini-Hochberg false-discovery-rate procedure.

In [29]:
N = len(informative_gene_ids)

fisher_validation = []

for condition in conditions:

    our_genes = our_persistent_sets[condition]
    published_genes = published_recovery_sets[condition]

    if published_genes is None:
        continue

    shared_genes = our_genes & published_genes

    a = len(shared_genes)
    b = len(our_genes - published_genes)
    c = len(published_genes - our_genes)
    d = N - a - b - c

    odds_ratio, p_value = fisher_exact(
        [[a, b],
         [c, d]],
        alternative="greater"
    )

    expected_overlap = (                                                       # calculating the expected random overlap
        len(our_genes)
        * len(published_genes)
        / N
    )

    fisher_validation.append(                                                  # one result per row condition
        {
            "condition": condition,
            "our_persistent_genes": len(our_genes),
            "published_recovery_degs": len(published_genes),
            "shared_genes": a,
            "expected_overlap_by_chance": expected_overlap,
            "overlap_enrichment": (
                a / expected_overlap
                if expected_overlap > 0
                else np.nan
            ),
            "odds_ratio": odds_ratio,
            "p_value": p_value,
        }
    )

fisher_validation = pd.DataFrame(
    fisher_validation
)

In [30]:
fisher_validation["fdr_bh"] = multipletests(                             # correcting the fisher test P-values for multiple testing
    fisher_validation["p_value"],
    method="fdr_bh"                                                      # Benjamini-Hochberg FDR correction
)[1]

fisher_validation

,condition,our_persistent_genes,published_recovery_degs,shared_genes,expected_overlap_by_chance,overlap_enrichment,odds_ratio,p_value,fdr_bh
0,cold,71,5,2,0.014,147.144,251.633,7.248e-05,1.812e-04
1,drought,1240,567,348,26.919,12.927,43.928,4.941e-324,2.470e-323
2,heat_drought,344,3,2,0.040,50.616,150.719,5.144e-04,8.573e-04
3,high_humidity,28,2,1,0.002,466.393,966.259,2.143e-03,2.679e-03
4,high_light,779,2,1,0.060,16.764,32.568,5.876e-02,5.876e-02


### **Insight —** Enrichment of Published Recovery DEGs Among Exploratory Persistent Genes

#### Published recovery DEGs were enriched among the genes classified as persistent in the exploratory analysis. The strongest evidence came from drought: 348 genes were shared between the 1,240 exploratory persistent genes and 567 published recovery DEGs, compared with only about 27 genes expected by chance. This corresponds to approximately 12.9-fold enrichment, and the overlap remained highly significant after FDR correction.

#### Significant enrichment was also found for cold, heat+drought, and high humidity, although the published recovery-DEG sets for these conditions were very small, so their large enrichment values should be interpreted cautiously. High light had one shared gene among two published recovery DEGs, but the overlap was not significant after FDR correction.

#### Overall, the published differential-expression results support the main pattern identified by the exploratory persistence analysis, especially the strong persistence observed after drought. Because both analyses are based on the same experiment, this should be treated as methodological validation rather than independent biological replication.

### Loading the Exploratory Persistent-Gene Table

#### The persistent-gene table generated during the exploratory analysis was loaded to provide gene-level information about the persistence calls used in the current validation.

In [31]:
PERSISTENT_GENE_TABLE_PATH = Path(
    "../data/interim/exploratory_analysis/persistent_gene_table.csv"
)

persistent_gene_table = pd.read_csv(
    PERSISTENT_GENE_TABLE_PATH
)

persistent_gene_table.shape

(3046, 7)

In [32]:
persistent_gene_table.columns.tolist()

['gene_id',
 'condition',
 'direction',
 'endpoint_shift',
 'recovery_shift',
 'reversibility_index',
 'persistence_breadth']

In [33]:
persistent_gene_table.head()

,gene_id,condition,direction,endpoint_shift,recovery_shift,reversibility_index,persistence_breadth
0,AT1G02920,cold,up,0.946,1.039,-0.098,6
1,AT1G07050,cold,up,2.540,0.401,0.842,2
2,AT1G09350,cold,up,1.763,0.449,0.745,1
3,AT1G09932,cold,up,0.400,0.398,0.005,4
4,AT1G10070,cold,up,0.542,0.356,0.343,6


### Standardizing Published Recovery DEG Results for Direction Comparison

#### The five published recovery DEG worksheets were combined into a single standardized table, and the sign of the published recovery log2FC was used to represent the direction of expression relative to control. Published recovery DEGs were then matched to exploratory persistent genes using both `gene_id` and stress condition.

#### Direction concordance was calculated only among genes identified by both approaches. For each condition, the analysis records how many shared genes have the same above-control or below-control direction in the exploratory persistence analysis and the published formal differential-expression result

In [34]:
published_recovery_long = []

for _, row in published_de_manifest[
    published_de_manifest["state"] == "recovery"
].iterrows():

    sheet = row["sheet_name"]
    condition = row["condition"]

    df = published_de_tables[sheet].copy()

    df["condition"] = condition

    df["published_direction"] = np.where(
        df["log2(FC)"] > 0,
        "up",
        "down"
    )

    published_recovery_long.append(
        df[
            [
                "gene_ID",
                "condition",
                "log2(FC)",
                "adj-P",
                "published_direction",
            ]
        ]
    )

published_recovery_long = pd.concat(
    published_recovery_long,
    ignore_index=True
)

published_recovery_long = (
    published_recovery_long
    .rename(
        columns={
            "gene_ID": "gene_id",
            "log2(FC)": "published_log2fc",
            "adj-P": "published_adj_p",
        }
    )
)

published_recovery_long.shape

(579, 5)

In [35]:
published_recovery_long.head()

,gene_id,condition,published_log2fc,published_adj_p,published_direction
0,AT1G25400,high_light,-1.064,1.034e-02,down
1,AT1G30250,high_light,1.004,1.732e-02,up
2,AT3G21530,high_humidity,-1.190,4.059e-03,down
3,AT2G43570,high_humidity,-1.137,2.950e-02,down
4,AT3G44300,drought,-3.904,3.038e-25,down


### Merging the Exploratory Persistent Genes to Published Recovery DEGs

In [36]:
direction_validation = persistent_gene_table.merge(
    published_recovery_long,
    on=[
        "gene_id",
        "condition",
    ],
    how="inner",
    validate="one_to_one",
)

direction_validation[
    [
        "gene_id",
        "condition",
        "direction",
        "recovery_shift",
        "published_log2fc",
        "published_direction",
    ]
].head()

,gene_id,condition,direction,recovery_shift,published_log2fc,published_direction
0,AT1G21310,cold,up,1.053,-1.271,down
1,AT3G22600,cold,up,0.980,-1.304,down
2,AT1G01680,drought,up,1.204,-3.595,down
3,AT1G02470,drought,up,0.695,-1.755,down
4,AT1G02850,drought,up,1.322,-2.181,down


In [37]:
direction_validation["direction_agrees"] = (
    direction_validation["direction"]
    == direction_validation["published_direction"]
)

### Summarizing Direction Concordance by Stress Condition

In [38]:
direction_summary = (
    direction_validation
    .groupby("condition")
    .agg(
        shared_genes=(
            "gene_id",
            "size",
        ),
        direction_agreements=(
            "direction_agrees",
            "sum",
        ),
    )
    .reset_index()
)

# Direction concordance = number of direction agreements / number of genes shared by both methods

direction_summary["direction_concordance"] = (                                 # calculating the direction concordance
    direction_summary["direction_agreements"]
    / direction_summary["shared_genes"]
)

direction_summary

,condition,shared_genes,direction_agreements,direction_concordance
0,cold,2,0,0.0
1,drought,348,0,0.0
2,heat_drought,2,0,0.0
3,high_humidity,1,0,0.0
4,high_light,1,0,0.0


### **Insight -** Published Contrast Orientation

#### Initial comparison of expression direction showed no direct sign agreement between the exploratory recovery differences and the published recovery-DEG log2(FC) values. 

#### Inspection of the shared gene-condition records showed that the signs were consistently reversed, indicating that the supplementary DEG workbook uses the opposite contrast orientation to the recovery-minus-control convention used in this project.

#### The original published log2(FC) values were retained unchanged. A separate sign-reoriented column was created only for directional comparison with the exploratory results.

### Verifying and Reorienting the Published Recovery Fold-Change Direction

#### Initial comparison showed that the exploratory recovery differences and the original published recovery log2(FC) values had systematically opposite signs. This was checked directly across the shared gene-condition records and supported by the negative numerical relationship between the two measures.

#### The original published fold-change values were retained unchanged. A separate sign-reoriented fold-change column was created so that the published results could be expressed in the same treatment-versus-control orientation used by the exploratory analysis. Direction concordance was then recalculated using this reoriented value.

### Checking Whether Every Shared Gene Has an Opposite Sign

In [39]:
direction_validation["sign_is_opposite"] = (
    np.sign(direction_validation["recovery_shift"])
    == -np.sign(direction_validation["published_log2fc"])
)

direction_validation["sign_is_opposite"].value_counts()

sign_is_opposite
True    354
Name: count, dtype: int64

#### Every shared gene does have an opposite sign.

In [40]:
direction_validation[
    [
        "recovery_shift",
        "published_log2fc"
    ]
].corr()

,recovery_shift,published_log2fc
recovery_shift,1.000,-0.518
published_log2fc,-0.518,1.000


In [41]:
# Preserving the original published values and creating a reoriented copy

published_recovery_long[
    "published_log2fc_treatment_vs_control"
] = -published_recovery_long["published_log2fc"]

# Converting the reoriented fold change into direction labels

published_recovery_long[
    "published_direction_treatment_vs_control"
] = np.where(
    published_recovery_long[
        "published_log2fc_treatment_vs_control"
    ] > 0,
    "up",
    "down"
)

### Rebuilding the Shared Gene-Level Direction Table

In [42]:
direction_validation = persistent_gene_table.merge(
    published_recovery_long,
    on=[
        "gene_id",
        "condition",
    ],
    how="inner",
    validate="one_to_one",
)

direction_validation["direction_agrees"] = (                                       # testing the direction agreement
    direction_validation["direction"]
    == direction_validation[
        "published_direction_treatment_vs_control"
    ]
)

### Summarizing Direction Concordance by Stress Condition

In [43]:
direction_summary = (
    direction_validation
    .groupby("condition")
    .agg(
        shared_genes=("gene_id", "size"),
        direction_agreements=(
            "direction_agrees",
            "sum",
        ),
    )
    .reset_index()
)

direction_summary["direction_concordance"] = (                                   # calculating the concordance fraction
    direction_summary["direction_agreements"]
    / direction_summary["shared_genes"]
)

direction_summary

,condition,shared_genes,direction_agreements,direction_concordance
0,cold,2,2,1.0
1,drought,348,348,1.0
2,heat_drought,2,2,1.0
3,high_humidity,1,1,1.0
4,high_light,1,1,1.0


### **Insight —** Directional Agreement with Published Differential Expression

#### All 354 gene-condition records shared between the exploratory persistent sets and the published recovery-DEG sets had opposite signs when the original published log2(FC) values were compared directly with the recovery shifts used in this project. This consistent inversion shows that the supplementary DEG results use the opposite contrast orientation from the treatment-versus-control convention used here.

#### After preserving the original published values and reorienting a separate copy for comparison, all 354 shared records agreed in expression direction. Drought provided most of this evidence, with complete directional agreement across all 348 shared genes.

#### The reoriented published fold changes were also moderately positively correlated with the exploratory recovery shifts (r ≈ 0.52). Exact values are not expected to match because the exploratory shifts were calculated from rlog-transformed expression values, whereas the published fold changes were estimated using a formal differential-expression model.

#### Together with the significant gene-set enrichment, this complete directional agreement shows that the exploratory persistence analysis captures a biological signal that is highly consistent with the published differential-expression results from the same experiment.

### Exporting The Published Differential-Expression Validation Results

#### The main intermediate and validation results from the published differential-expression comparison were exported to a dedicated `published_de_validation` directory. These files preserve workbook validation, condition-level DEG counts, gene-set overlap, Fisher enrichment and FDR results, informative-gene coverage, standardized published recovery results, and gene-level and condition-level direction validation.

#### Original published values are retained unchanged in the exported data, including the original fold-change orientation. Reoriented values used for directional comparison are stored separately so that the transformation remains explicit and reproducible.

In [44]:
PUBLISHED_DE_VALIDATION_DIR = Path(
    "../data/interim/published_de_validation"
)

PUBLISHED_DE_VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [45]:
published_de_summary.to_csv(                                            # published DEG workbook summary
    PUBLISHED_DE_VALIDATION_DIR
    / "published_de_workbook_summary.csv",
    index=False
)

published_de_counts.to_csv(                                             # published DEG counts by condition
    PUBLISHED_DE_VALIDATION_DIR
    / "published_de_condition_counts.csv",
    index=False
)

published_de_validation.to_csv(                                         # exploratory-persistence and published-DEG overlap
    PUBLISHED_DE_VALIDATION_DIR
    / "persistent_vs_published_recovery_degs.csv",
    index=False
)

fisher_validation.to_csv(                                               # published recovery-DEG enrichment results
    PUBLISHED_DE_VALIDATION_DIR
    / "published_recovery_overlap_enrichment.csv",
    index=False
)

direction_validation.to_csv(                                            # gene-level direction validation
    PUBLISHED_DE_VALIDATION_DIR
    / "published_recovery_direction_validation.csv",
    index=False
)

direction_summary.to_csv(                                               #  direction concordance summary
    PUBLISHED_DE_VALIDATION_DIR
    / "published_recovery_direction_summary.csv",
    index=False
)

### **Integrated Interpretation**

#### The published differential-expression results provide a formal statistical check on the recovery and persistence patterns identified earlier in this project.

#### The clearest agreement occurs under drought. The exploratory analysis identified 1,240 genes with persistent same-direction expression after recovery, while the original study reported 567 significant recovery DEGs. Of these, 348 genes were shared between the two approaches. Only about 27 shared genes would be expected by chance given the common 26,118-gene background, corresponding to approximately 12.9-fold overlap enrichment. This overlap remained highly significant after multiple-testing correction.

#### Significant enrichment was also observed for cold, heat+drought, and high humidity. However, the published recovery-DEG sets for these conditions were very small, so their large enrichment values should be interpreted cautiously. High light showed one shared gene among two published recovery DEGs but did not remain significant after FDR correction.

#### The two approaches are not expected to identify identical gene sets. The exploratory persistence framework uses rlog-transformed expression values and a control-derived empirical response floor to describe residual expression patterns, whereas the published analysis uses a formal count-based differential-expression model together with fold-change and adjusted-p-value criteria. The exploratory framework therefore captures a broader set of residual expression changes, while the published DEG analysis applies stricter statistical requirements.

#### A contrast-orientation check showed that the published recovery log2(FC) values used the opposite sign convention from the treatment-versus-control orientation used in this project. The original published values were retained unchanged, and a separate reoriented value was used for comparison. After this adjustment, all 354 shared gene-condition records agreed in expression direction. The reoriented published fold changes were also moderately positively correlated with the exploratory recovery shifts (r ≈ 0.52). Exact magnitudes are not expected to match because the two quantities are derived using different analytical procedures.

#### Together, the enrichment results and complete directional agreement show that the exploratory persistence framework captures a recovery-stage transcriptional signal that is highly consistent with the published differential-expression analysis from the same experiment. The strongest evidence comes from drought, where both approaches identify a substantial and directionally consistent residual expression response.

### **Limitations**

#### This validation uses the published filtered DEG lists rather than the complete DESeq2 output for all tested genes. Genes absent from the supplementary DEG tables therefore cannot be assigned their published fold changes, p-values, or adjusted p-values. The analysis can compare the exploratory results with genes that passed the authors' DEG criteria, but it cannot examine the full distribution of published differential-expression statistics.

#### The two approaches also define residual response differently. The exploratory persistence framework was constructed from processed rlog expression values using a control-derived empirical response floor, whereas the published analysis used raw count data and formal differential-expression modeling together with fold-change and adjusted-p-value thresholds. Agreement between the methods therefore shows consistency in the detected signal, while disagreement does not by itself indicate that either approach is incorrect.

#### Exact recovery DEG lists are available for only five of the nine stress conditions. For the remaining conditions, the recovery contrasts were performed and the study reports fewer than 10 significant recovery DEGs, but their exact counts and gene identities are not available from the supplementary workbook. Gene-level overlap, enrichment, and direction testing therefore cannot be performed for those conditions.

#### The validation is also based on the same underlying experiment used to generate the exploratory expression matrix. It therefore provides methodological validation against the study's formal differential-expression analysis, but it does not constitute independent biological replication. Confirmation in an independent and suitably comparable dataset would provide stronger evidence that the observed persistence patterns generalize beyond this experiment.

#### Finally, the supplementary workbook uses the opposite fold-change orientation from the convention used in this project. The original published values were preserved unchanged, and a separate sign-reoriented variable was created only for directional comparison. This transformation affects the sign convention only and does not alter the magnitude or statistical significance of the published results.

### **Conclusion**

#### The published differential-expression analysis supports the major recovery pattern identified by the exploratory persistence framework.

#### Drought showed the strongest overall convergence between the two approaches, with 348 genes shared between the exploratory persistent set and the published recovery-DEG set. This represented approximately 12.9-fold more overlap than expected by chance and remained highly significant after multiple-testing correction. After the published contrast orientation was harmonized with the convention used in this project, all shared gene-condition records agreed in expression direction.

#### These results show that the persistence patterns are strongly consistent with the formal count-based differential-expression analysis and are not simply a consequence of the exploratory response threshold. At the same time, the broader exploratory framework and the stricter published DEG analysis describe different aspects of recovery and should not be treated as interchangeable definitions.

#### Overall, this validation supports the use of the persistence framework for cross-stress comparison and biological interpretation, while maintaining a clear distinction between exploratory expression persistence and formal differential-expression significance. Independent biological replication would still be required to determine how well these patterns generalize beyond the original experiment.